In [ ]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel
import torch
import re
import gdown
from typing import List, Tuple, Any
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

## Buscamos datos, los limpiamos y armamos un dataframe para poder trabajar

In [ ]:
from pathlib import Path
path = Path.home() / "Desktop" / "datasets" / "es.txt"
data = pd.read_table(path, header=None, encoding="utf-8")

data.columns = ['text']
display(data.head())

In [ ]:
path = Path.home() / "Desktop" / "datasets" / "spa_sentences.tsv"
data_1 = pd.read_table(path, header=None, encoding="utf-8")

#data_1 = pd.read_table(output_path, header=None)
data_1 = data_1[[2]]
data_1.columns = ['text']

# concateno los datasets
data = pd.concat([data, data_1], axis=0)
display(data.head())

Limpiamos la base de datos usando expresiones regulares. Eliminamos aclaraciones de los subtitulos como "(Aplausos) (Risas)" y signos de puntuación que no nos interesa predecir.

In [ ]:
data['text'] = data['text'].apply(lambda x: re.sub(r'\s*\([^)]*\)', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'[^\w¿?.,\s]+', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())
data['text'] = data['text'].apply(lambda x: re.sub(r'\s(\.\.\.|\.\.)\s', ' ', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s(\.\.\.|\.\.)$', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'^(\.\.\.|\.\.)\s', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\.\s', ' ', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\.$', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'^\.\s', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s,\s', ' ', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s,$', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'^,\s', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\?\s', ' ', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\?$', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'^\?\s', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s¿\s', ' ', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s¿$', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'^¿\s', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s""', '', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\.', '.', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s,', ',', x))
data['text'] = data['text'].apply(lambda x: re.sub(r'\s\?', '?', x))

In [ ]:
#separamos el dataset en train y validacion
from sklearn.model_selection import train_test_split
desarrollo , heldout = train_test_split(data, test_size = 0.1 , random_state = 33, shuffle = True)
train, test = train_test_split(desarrollo, test_size = 0.1 , random_state = 33, shuffle = True)

In [ ]:
# Cargamos el tokenizador pre-entrenado multilingual
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

# Procesamiento de datos para Random Forest
Generamos variables a partir de las existentes


In [ ]:
def get_capitalization_label(word: str) -> int:
    """Determina la etiqueta de capitalización (0-3) según la palabra original."""
    if word.islower():
        return 0  # todo minúscula
    elif word.istitle():
        return 1  # Primera mayúscula
    elif word.isupper():
        return 3  # TODO MAYÚSCULA
    else:
        return 2  # Mixto (ej: YouTube, McDonald)

def extract_punctuation(word: str) -> Tuple[str, str, str]:
    """
    Separa la puntuación inicial y final de la palabra limpia.
    Retorna: (puntuación_inicial, puntuación_final, palabra_limpia)
    """
    punct_start = ""
    punct_end = ""
    clean_word = word

    # Detectar inicio
    if clean_word.startswith('¿'):
        punct_start = '¿'
        clean_word = clean_word[1:] # Quitamos el signo

    # Detectar final
    if clean_word.endswith('?'):
        punct_end = '?'
        clean_word = clean_word[:-1]
    elif clean_word.endswith('.'):
        punct_end = '.'
        clean_word = clean_word[:-1]
    elif clean_word.endswith(','):
        punct_end = ','
        clean_word = clean_word[:-1]

    # Limpieza extra de seguridad (lower)
    clean_word = clean_word.lower()

    return punct_start, punct_end, clean_word

def process_single_word(original_word: str,
                        n_sentence: int,
                        n_word_idx: int,
                        tokenizer: Any) -> dict:
    """
    Tokeniza una palabra y expande sus metadatos para alinearlos con los sub-tokens.
    """
    # 1. Extraer características básicas
    punct_start, punct_end, clean_word = extract_punctuation(original_word)
    cap_label = get_capitalization_label(original_word)

    # 2. Tokenización BERT
    # Si la palabra quedó vacía tras limpiar puntuación, la ignoramos (o manejamos según caso)
    if not clean_word:
        return None

    tokens = tokenizer.tokenize(clean_word)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    n_subtokens = len(ids)

    if n_subtokens == 0:
        return None

    # 3. Distribución de etiquetas (Logic Alignment)
    # Capitalización: Se repite para todos los sub-tokens
    list_cap = [cap_label] * n_subtokens

    # Índices: Se repiten para todos
    list_n_sent = [n_sentence] * n_subtokens
    list_n_word = [n_word_idx] * n_subtokens

    # Puntuación Inicial: Solo al PRIMER sub-token, el resto vacío
    list_punct_start = [""] * n_subtokens
    list_punct_start[0] = punct_start

    # Puntuación Final: Solo al ÚLTIMO sub-token, el resto vacío
    list_punct_end = [""] * n_subtokens
    list_punct_end[-1] = punct_end

    return {
        'token': tokens,
        'id': ids,
        'y_puntuacion_inicial': list_punct_start,
        'y_puntuacion_final': list_punct_end,
        'y_capitalizacion': list_cap,
        'n_sentence': list_n_sent,
        'n_word_in_sentence': list_n_word
    }

In [ ]:
def create_bert_dataset(df_raw: pd.DataFrame, tokenizer: Any) -> pd.DataFrame:
    """
    Función principal: Transforma texto crudo en un DataFrame tokenizado para BERT.
    """
    # Acumuladores (Diccionario de listas es más rápido que append a muchas variables)
    data_dict = {
        'token': [], 'id': [],
        'y_puntuacion_inicial': [], 'y_puntuacion_final': [], 'y_capitalizacion': [],
        'n_sentence': [], 'n_word_in_sentence': []
    }

    print("Procesando oraciones y tokenizando...")

    # Iteramos sobre el DataFrame original
    # Asumimos que la columna 0 tiene el texto
    text_col = df_raw.columns[0]

    for idx_sentence, row in tqdm(df_raw.iterrows(), total=len(df_raw)):
        sentence = str(row[text_col])
        words = sentence.split()

        for idx_word, original_word in enumerate(words):

            # Procesamos cada palabra con la función auxiliar
            result = process_single_word(original_word, idx_sentence, idx_word, tokenizer)

            if result:
                # Extendemos las listas principales
                for key in data_dict:
                    data_dict[key].extend(result[key])

    # Crear DataFrame final
    return pd.DataFrame(data_dict)

def add_relative_position_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega la longitud de la oración y la posición relativa del token.
    """
    df_out = df.copy()

    # Calcular longitud máxima por oración (transform difunde el resultado a todas las filas)
    # Esto evita el merge explícito, es más directo.
    df_out['sentence_length'] = df_out.groupby('n_sentence')['n_word_in_sentence'].transform('max')

    # Calcular posición relativa (smoothing +1 para evitar división por cero si fuera necesario)
    # Nota: Tu lógica original dividía n_word / len.
    # Si n_word va de 0 a 10 y len es 10, el último es 10/11 = 0.9. Correcto.
    df_out['relative_position'] = df_out['n_word_in_sentence'] / (df_out['sentence_length'] + 1)

    return df_out

In [ ]:
df_train_rf = create_bert_dataset(df_raw=train, tokenizer=tokenizer)
df_train_rf = add_relative_position_features(df_train_rf)

df_eval_rf = create_bert_dataset(df_raw=test, tokenizer=tokenizer)
df_eval_rf = add_relative_position_features(df_eval_rf)

df_heldout_rf = create_bert_dataset(df_raw=heldout, tokenizer=tokenizer)
df_heldout_rf = add_relative_position_features(df_heldout_rf)

Armamos un dataframe para entrenar un random forest. Cómo este tipo de algortimo de aprendizaje ve la información de un token por vez, vamos a generar columnas que den información de su contexto a partir de las representaciones vectoriales de BERT.


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)


In [ ]:
model.eval()
# Función proporcionada por los docentes para extraer embeddings a partir
# de tokens
def get_multilingual_token_embedding(token_id):
  """
  Devuelve el embedding (estático) para el token.
  """
  if token_id is None or token_id == tokenizer.unk_token_id:
    print(f"❌ El token '{token_id}' no pertenece al vocabulario de multilingual BERT.")
    return None
  embedding_vector = model.embeddings.word_embeddings.weight[token_id]
  return embedding_vector

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
import os
import sys
import gc

def generar_embeddings_en_disco(df, col_id, nombre_archivo, model, tokenizer, batch_size=512):
    """
    Genera embeddings estáticos usando memmap en disco D:, aplicando la lógica
    de filtrado de UNK e IDs inválidos.
    """

    # --- CONFIGURACIÓN ---
    DISCO_DESTINO = 'C:/'
    CARPETA = 'Proyecto_Embeddings'
    EMBEDDING_DIM = 768
    dtype = np.float16

    # Rutas
    save_dir = os.path.join(DISCO_DESTINO, CARPETA)
    full_path = os.path.join(save_dir, nombre_archivo)
    os.makedirs(save_dir, exist_ok=True)

    # Limpieza previa de memoria
    gc.collect()
    torch.cuda.empty_cache()

    # --- VALIDACIÓN VOCABULARIO ---
    try:
        UNK_ID = tokenizer.unk_token_id
        VOCAB_SIZE = model.config.vocab_size
    except:
        UNK_ID = 100
        VOCAB_SIZE = 119547
        print("⚠️ Usando configuración mBERT por defecto.")

    # --- PREPARAR MEMMAP ---
    total_samples = len(df)
    bytes_needed = total_samples * EMBEDDING_DIM * np.dtype(dtype).itemsize

    print(f"\n📂 Procesando: {nombre_archivo}")
    print(f"   Filas: {total_samples}")
    print(f"   Espacio reservado: {bytes_needed / (1024**3):.2f} GB")

    # Crear/Resetear archivo
    with open(full_path, 'wb') as f:
        f.seek(bytes_needed - 1)
        f.write(b'\0')

    # Mapear
    all_embeddings = np.memmap(full_path, dtype=dtype, mode='r+', shape=(total_samples, EMBEDDING_DIM))

    # --- PROCESAMIENTO ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()

    print("🚀 Iniciando generación...")

    for i in tqdm(range(0, total_samples, batch_size), desc=f"Generando {nombre_archivo}"):

        # A. Cargar Batch
        batch_ids_raw = df[col_id].iloc[i : i + batch_size].fillna(UNK_ID).tolist()

        # B. Sanitización (Lógica de Profesores)
        clean_ids = []
        mask_invalidos = []

        for pid in batch_ids_raw:
            try:
                pid_int = int(pid)
            except:
                pid_int = UNK_ID

            # Chequeos
            if pid_int < 0 or pid_int >= VOCAB_SIZE: # Fuera de rango
                clean_ids.append(UNK_ID)
                mask_invalidos.append(True)
            elif pid_int == UNK_ID: # Es UNK explícito
                clean_ids.append(UNK_ID)
                mask_invalidos.append(True)
            else:
                clean_ids.append(pid_int)
                mask_invalidos.append(False)

        # C. GPU
        input_ids = torch.tensor(clean_ids, dtype=torch.long).unsqueeze(1).to(device)

        with torch.no_grad():
            batch_output = model.embeddings.word_embeddings(input_ids)
            batch_output = batch_output.squeeze(1)

        # D. Filtro de Ceros y Guardado
        batch_numpy = batch_output.detach().cpu().numpy().astype(dtype)

        mask_array = np.array(mask_invalidos)
        if mask_array.any():
            batch_numpy[mask_array] = 0.0

        # E. Escribir en Memmap
        end_idx = min(i + batch_size, total_samples)
        all_embeddings[i : end_idx] = batch_numpy

        if i % (batch_size * 50) == 0:
            all_embeddings.flush()

    all_embeddings.flush()
    print(f"✅ Guardado exitoso en: {full_path}")

    # Liberar la variable memmap para soltar el archivo
    del all_embeddings
    gc.collect()

In [ ]:
# --- 1. PROCESAR TRAIN ---
generar_embeddings_en_disco(
    df=df_train_rf,
    col_id='id',
    nombre_archivo='embeddings_train_safe.dat',
    model=model,
    tokenizer=tokenizer
)

# --- 2. PROCESAR EVAL (Validación / Dev) ---
generar_embeddings_en_disco(
    df=df_eval_rf,
    col_id='id',
    nombre_archivo='embeddings_eval_safe.dat', # Ojo al nombre
    model=model,
    tokenizer=tokenizer
)

# --- 3. PROCESAR HELDOUT (Test Final) ---
generar_embeddings_en_disco(
    df=df_heldout_rf,
    col_id='id',
    nombre_archivo='embeddings_heldout_safe.dat', # Ojo al nombre
    model=model,
    tokenizer=tokenizer
)

print("\n🎉 ¡TODOS LOS EMBEDDINGS HAN SIDO GENERADOS EN EL DISCO D! 🎉")

In [ ]:
df_train_rf.columns

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import os
import gc

def calcular_features_coseno(ruta_archivo, total_samples, batch_size=10000):
    """
    Calcula la distancia coseno con el token siguiente y anterior
    leyendo embeddings desde disco.
    Retorna: (dist_prev, dist_next) como arrays numpy float32.
    """
    EMBEDDING_DIM = 768
    DTYPE_IN = np.float16

    print(f"\n📐 Calculando cosenos para: {os.path.basename(ruta_archivo)}")
    print(f"   Filas: {total_samples}")

    # 1. Conectar al disco (Lectura)
    X_memmap = np.memmap(ruta_archivo, dtype=DTYPE_IN, mode='r', shape=(total_samples, EMBEDDING_DIM))

    # 2. Calcular Normas (Magnitud)
    norms = np.zeros(total_samples, dtype=np.float32)

    # Procesar normas por batch
    for i in range(0, total_samples, batch_size):
        end = min(i + batch_size, total_samples)
        batch = X_memmap[i:end].astype(np.float32)
        norms[i:end] = np.linalg.norm(batch, axis=1)

    # Evitar división por cero
    norms[norms == 0] = 1e-10

    # 3. Calcular Distancia NEXT
    dist_next = np.zeros(total_samples, dtype=np.float32)

    # Iteramos hasta el anteúltimo (porque el último no tiene siguiente)
    for i in tqdm(range(0, total_samples - 1, batch_size), desc="Distancia Next"):
        start = i
        end = min(i + batch_size, total_samples - 1)

        # Batch actual y siguiente
        batch_curr = X_memmap[start:end].astype(np.float32)
        batch_next = X_memmap[start+1 : end+1].astype(np.float32)

        # Coseno
        dot_product = np.sum(batch_curr * batch_next, axis=1)
        denom = norms[start:end] * norms[start+1 : end+1]
        cosine_sim = dot_product / denom

        # Distancia
        dist = 1.0 - np.clip(cosine_sim, -1.0, 1.0)
        dist_next[start:end] = dist

    # 4. Calcular Distancia PREV (Shift)
    dist_prev = np.zeros(total_samples, dtype=np.float32)
    dist_prev[1:] = dist_next[:-1] # El prev del indice 1 es el next del indice 0
    dist_prev[0] = 1.0 # El primero no tiene previo (asumimos distancia máxima)

    # Limpieza
    del X_memmap
    del norms
    gc.collect()

    return dist_prev, dist_next

In [ ]:
# Configuración de rutas (Asegúrate que coinciden con los nombres de la celda anterior)
DIR_BASE = 'C:/Proyecto_Embeddings'

configs = [
    {
        'nombre': 'TRAIN',
        'df': df_train_rf,
        'archivo': 'embeddings_train_safe.dat'
    },
    {
        'nombre': 'EVAL',
        'df': df_eval_rf,
        'archivo': 'embeddings_eval_safe.dat'
    },
    {
        'nombre': 'HELDOUT',
        'df': df_heldout_rf,
        'archivo': 'embeddings_heldout_safe.dat'
    }
]

# Diccionario para guardar tus features calculadas
# Estructura: 'TRAIN': {'prev': array, 'next': array}, ...
features_coseno = {}

for cfg in configs:
    full_path = os.path.join(DIR_BASE, cfg['archivo'])
    n_samples = len(cfg['df'])

    # Llamamos a la función
    d_prev, d_next = calcular_features_coseno(full_path, n_samples)

    # Guardamos en el diccionario global
    features_coseno[cfg['nombre']] = {
        'prev': d_prev,
        'next': d_next
    }

print("\n✅ ¡Cálculos de coseno terminados para todos los datasets!")

In [ ]:
import numpy as np
import pandas as pd
import os
import gc
from tqdm.auto import tqdm

def fusionar_features_y_embeddings(df, ruta_embeddings, features_coseno, ruta_salida):
    """
    Fusiona:
    1. Embeddings (desde disco .dat)
    2. Features manuales (n_word_in_sentence, is_subtoken)
    3. Features coseno (prev, next)

    Genera un archivo único .dat en ruta_salida.
    """
    # --- CONFIGURACIÓN ---
    N_SAMPLES = len(df)
    DIM_BERT = 768
    DIM_MANUAL = 4 # pos, is_sub, prev, next
    DIM_TOTAL = DIM_BERT + DIM_MANUAL
    DTYPE = np.float16

    print(f"\n🏗️ Fusionando dataset: {os.path.basename(ruta_salida)}")
    print(f"   Filas: {N_SAMPLES} | Columnas Finales: {DIM_TOTAL}")

    # --- 1. PREPARAR FEATURES MANUALES (RAM) ---
    print("   -> Generando features manuales en memoria...")

    # A. Posición (n_word_in_sentence)
    pos = df['n_word_in_sentence'].values.astype(np.float32)

    # B. Is Subtoken (Calculado al vuelo desde la columna 'token')
    # Convertimos a string por seguridad y chequeamos '##'
    is_sub = df['token'].astype(str).str.startswith('##').values.astype(np.float32)

    # C. Cosenos (Vienen pasados como argumento)
    d_prev = features_coseno['prev']
    d_next = features_coseno['next']

    # Stack en una matriz temporal (N, 4)
    # Convertimos todo a float16 AHORA para que coincida con el memmap de salida
    X_manual = np.column_stack((pos, is_sub, d_prev, d_next)).astype(DTYPE)

    # --- 2. CREAR ARCHIVO FINAL (DISCO D) ---
    # Si existe lo sobreescribe
    X_final = np.memmap(ruta_salida, dtype=DTYPE, mode='w+', shape=(N_SAMPLES, DIM_TOTAL))

    # --- 3. COPIAR EMBEDDINGS (Chunks) ---
    print("   -> Copiando Embeddings desde disco...")

    # Conectamos al archivo de embeddings origen (Solo lectura)
    emb_reader = np.memmap(ruta_embeddings, dtype=DTYPE, mode='r', shape=(N_SAMPLES, DIM_BERT))

    BATCH_SIZE = 50000
    for i in tqdm(range(0, N_SAMPLES, BATCH_SIZE), desc="Copiando"):
        end = min(i + BATCH_SIZE, N_SAMPLES)

        # Copiar parte BERT (Cols 0 a 768)
        X_final[i:end, :DIM_BERT] = emb_reader[i:end]

        # Forzar escritura
        if i % (BATCH_SIZE * 5) == 0:
            X_final.flush()

    # --- 4. COPIAR FEATURES MANUALES ---
    print("   -> Agregando features manuales...")
    # Copiar parte Manual (Cols 768 a 772)
    X_final[:, DIM_BERT:] = X_manual[:]

    X_final.flush()
    print(f"   ✅ Archivo guardado: {ruta_salida}")

    # Limpieza
    del X_manual
    del emb_reader
    del X_final
    gc.collect()

In [ ]:
# Definimos nombres finales
DIR_BASE = 'C:/Proyecto_Embeddings'

# Mapeo de configuraciones (Asegúrate de tener features_coseno cargado del paso anterior)
# Reciclamos la lista 'configs' si la tienes, o la definimos de nuevo explícitamente:

lista_tareas = [
    {
        'key': 'TRAIN',
        'df': df_train_rf,
        'emb_in': 'embeddings_train_safe.dat',
        'file_out': 'dataset_train_completo.dat'
    },
    {
        'key': 'EVAL', # Esto es tu df_dev_rf / df_eval_rf
        'df': df_eval_rf,
        'emb_in': 'embeddings_eval_safe.dat',
        'file_out': 'dataset_eval_completo.dat'
    },
    {
        'key': 'HELDOUT',
        'df': df_heldout_rf,
        'emb_in': 'embeddings_heldout_safe.dat',
        'file_out': 'dataset_heldout_completo.dat'
    }
]

# --- BUCLE DE EJECUCIÓN ---
for tarea in lista_tareas:
    key = tarea['key']

    # Rutas completas
    path_emb_in = os.path.join(DIR_BASE, tarea['emb_in'])
    path_out = os.path.join(DIR_BASE, tarea['file_out'])

    # Recuperar features coseno del diccionario en memoria
    feats_cos = features_coseno[key]

    # Ejecutar fusión
    fusionar_features_y_embeddings(
        df=tarea['df'],
        ruta_embeddings=path_emb_in,
        features_coseno=feats_cos,
        ruta_salida=path_out
    )

print("\n🎉 ¡PROCESO DE INGENIERÍA DE DATOS TERMINADO!")
print("Ya tienes los 3 archivos .dat listos para entrenar y validar.")

In [ ]:
import numpy as np
import os

DIR_BASE = 'C:/Proyecto_Embeddings'

print("💾 Iniciando exportación de los 3 targets...")

# --- DEFINICIÓN DE NOMBRES ---
# Mapeamos: Nombre de columna -> Sufijo del archivo
# Ejemplo: y_train_cap.npy, y_train_ini.npy, y_train_fin.npy
targets_config = [
    {'col': 'y_capitalizacion',       'sufijo': '_cap', 'tipo': 'int'},
    {'col': 'y_puntuacion_inicial',   'sufijo': '_ini', 'tipo': 'str'},
    {'col': 'y_puntuacion_final',     'sufijo': '_fin', 'tipo': 'str'}
]

# Datasets disponibles en memoria
datasets = {
    'train': df_train_rf,
    'eval':  df_eval_rf,
    'heldout': df_heldout_rf
}

# --- BUCLE DE GUARDADO ---
for ds_name, df in datasets.items():
    print(f"\nProcesando dataset: {ds_name.upper()} ({len(df)} filas)")

    for config in targets_config:
        col_name = config['col']
        filename = f"y_{ds_name}{config['sufijo']}.npy"
        full_path = os.path.join(DIR_BASE, filename)

        # Extracción de datos
        data_values = df[col_name].values

        # Conversión de tipos para ahorrar espacio
        if config['tipo'] == 'int':
            # Capitalización: Enteros pequeños
            data_to_save = data_values.astype(np.int8)
        else:
            # Puntuación: Texto (Strings)
            # Aseguramos que sean strings puros y no objetos pandas raros
            data_to_save = data_values.astype(str)

        # Guardar
        np.save(full_path, data_to_save)
        print(f"   -> Guardado: {filename}")

print("\n✅ ¡LISTO! Se generaron 9 archivos .npy (3 por cada dataset).")
print("Ahora sí, ya puedes cerrar este notebook y liberar toda la RAM.")

In [ ]:
import numpy as np
import os

DIR_BASE = 'C:/Proyecto_Embeddings'

# Guardamos la columna n_sentence como enteros
print("💾 Guardando mapa de oraciones...")
np.save(os.path.join(DIR_BASE, 'n_sentence_train.npy'), df_train_rf['n_sentence'].values.astype(np.int32))
np.save(os.path.join(DIR_BASE, 'n_sentence_eval.npy'), df_eval_rf['n_sentence'].values.astype(np.int32))
# np.save(os.path.join(DIR_BASE, 'n_sentence_heldout.npy'), df_heldout_rf['n_sentence'].values.astype(np.int32))

print("✅ Mapa de oraciones guardado.")